In [4]:
# import tif file (file - single or stack; or import matrix)

# Works for 

import numpy as np
import os
from pathlib import Path
import json
import socket
import tifffile # !pip install tifffile
import platform
import re
from datetime import datetime

import pandas as pd
import tifffile # !pip install tifffile

# Path to your ScanImage stack
# path = '/Users/jcoleman/Documents/--LARGE DATA--/#Khodayari/260113_NK/process1/'
path = '/Users/jcoleman/Documents/--LARGE DATA--/#VonRoemeling/'
# path = '/Users/jcoleman/Documents/--LARGE DATA--/#Pizzi/TBI-sham rbc velocity/data/raw/htau pointscan data for analysis/mht01_ZSAW_points/ROIA_grpA/'
# # Option A: load entire stack into a single array
# #    tif_data will have shape (num_slices, height, width)

# tif_filename = 'fileCELL_3x_750nm_00004.tif'
tif_filename = 'T1800_25x3x_1100nm_area07_00001.tif'
# tif_filename = 'Zsaw_mht01_25x4x_800nm_grpA_00001.tif'

tif_file = os.path.join(
    path, 
    tif_filename
)

tif_data = tifffile.imread(tif_file)

print(tif_data.shape)


(1800, 2, 512, 512)


In [ ]:
# Works for 2-channel tif_data with 4 dims (T,C,Y,X)
# Extract channel 1 and channel 2 and reshape to (Z, Y, X)
channel_1_data = tif_data[:, 0, :, :]  # shape: (33, 512, 512)
channel_2_data = tif_data[:, 1, :, :]  # shape: (33, 512, 512)

print(f"Channel 1   shape: {channel_1_data.shape}")
print(f"Channel 2   shape: {channel_2_data.shape}")

# Transpose to (Y, X, Z) if needed for your code
channel_1_data = np.transpose(channel_1_data, (1, 2, 0))  # shape: (512, 512, 33)
channel_2_data = np.transpose(channel_2_data, (1, 2, 0))  # shape: (512, 512, 33)

print(f"Channel 1 reshape: {channel_1_data.shape}")
print(f"Channel 2 reshape: {channel_2_data.shape}")

# Sanity check the channels
import matplotlib.pyplot as plt

frame = 0
plt.figure()
plt.imshow(channel_1_data[:,:,frame], cmap='gray')
plt.title(f'Channel 1 (frame {frame})')
plt.show()

plt.figure()
plt.imshow(channel_2_data[:,:,frame], cmap='gray')
plt.title(f'Channel 2 (frame {frame})')
plt.show()

In [ ]:
# tif_file = path + 'Zsaw_mht01_25x4x_800nm_grpA_00001.tif'
# tif_data = tifffile.imread(tif_file)
tif_data = channel_1_data
# print(tif_data.shape)

# If you prefer (height, width, num_slices), just transpose:
# (ie if its not Loaded stack with shape (512, 512, 33))
# tif_data = np.transpose(tif_data, (1, 2, 0))

print(f"Loaded stack with shape {tif_data.shape}")
print(f"Number of slices (tif_data.shape[2]): {tif_data.shape[2]}")

In [ ]:
# Get TIF image resolution (px/cm) and FOV in microns (L x W)

import json
import numpy as np
import tifffile
import matplotlib.pyplot as plt

# --- 1) Read metadata from the first page ---------------------------
# Read tif file from "import tiff file"[o] section
#tif_file = 'Zsaw_mht01_25x4x_800nm_grpA_00001.tif'

with tifffile.TiffFile(tif_file) as tif:
    page = tif.pages[0]
    tags = page.tags

    # These come in as (num, denom) tuples
    res_y_raw = tags['YResolution'].value
    res_x_raw = tags['XResolution'].value
    resolution_y = res_y_raw[0] / res_y_raw[1]
    resolution_x = res_x_raw[0] / res_x_raw[1]
    resolution_unit = tags['ResolutionUnit'].value  # 1=no unit, 2=inches, 3=cm

    # 1) Image dimensions:
    height, width = page.shape # returns (rows, cols)
    # or equivalently:
    # width  = page.image_width
    # height = page.image_length

    # ScanImage sticks its full JSON header in the Artist tag:
    image_scan_area_metadata = tags['Artist'].value

# map the TIFF spec codes to human‐readable units
unit_map = {1: 'none', 2: 'inch', 3: 'cm'}
unit_name = unit_map.get(resolution_unit, 'unknown')

print(f"Resolution: {resolution_x:.1f} px/{unit_name}, {resolution_y:.1f} px/{unit_name}")

if unit_name == 'cm':
    # physical size in cm:
    width_cm  = width  / resolution_x
    height_cm = height / resolution_y
    # convert to microns:
    width_um  = width_cm  * 1e4
    height_um = height_cm * 1e4

elif unit_name == 'inch':
    # 1 in = 2.54 cm = 25 400 μm
    width_in  = width  / resolution_x
    height_in = height / resolution_y
    width_um  = width_in  * 25400
    height_um = height_in * 25400

else:
    raise ValueError(f"Unsupported resolution unit: {resolution_unit}")

print(f"Field-of-view: {width_um:.2f}×{height_um:.2f} μm")
# NOTE from 'info' in FIJI get:
#   ResolutionUnit = Centimeter
#   XResolution = 36446.21105868775
#   YResolution = 36446.21105868775

In [ ]:
print(image_scan_area_metadata) # this is the motherlode

In [ ]:
needles = ["hbeams", "pockels", "power", "beam", "lut", "fastz", "zstep", "zs", "stack", "motor"]
for n in needles:
    hits = [k for k in si_meta.keys() if n in k.lower()]
    print(n, len(hits))
    for k in hits[:30]:
        print("  ", k)

In [ ]:
# Sanity check for new Z-indices
from pathlib import Path
import tifffile as tiff
import re

def debug_scanimage_metadata(tif_path, max_chars=4000):
    tif_path = Path(tif_path)
    with tiff.TiffFile(tif_path) as tf:
        p0 = tf.pages[0]

        print("File:", tif_path)
        print("Pages:", len(tf.pages))
        print("Available tags on page 0:", sorted([t.name for t in p0.tags.values()]))

        desc_tag = p0.tags.get("ImageDescription")
        desc = desc_tag.value if desc_tag else ""
        print("\n--- ImageDescription (first chars) ---")
        print(desc[:max_chars] if desc else "<None>")

        # Grep-style search for likely Z-related lines if it's the classic "SI.key = value" text
        if desc:
            lines = desc.splitlines()
            hit = [ln for ln in lines if re.search(r"(hStackManager|hFastZ|stackZ|framesPerSlice|numSlices|\bzs\b|motor|zPos|piezo|focus)", ln, re.IGNORECASE)]
            print("\n--- Lines containing Z-related terms ---")
            for ln in hit[:200]:
                print(ln)

debug_scanimage_metadata(tif_file)


In [ ]:
import tifffile as tiff
import re

def dump_scanimage_tags(tif_path, max_chars=6000):
    with tiff.TiffFile(tif_path) as tf:
        p0 = tf.pages[0]
        plast = tf.pages[-1]

        def show(page, label):
            print(f"\n===== {label} =====")
            for tagname in ["Artist", "Software", "ImageDescription"]:
                tag = page.tags.get(tagname)
                val = tag.value if tag else None
                if val is None:
                    print(f"{tagname}: <None>")
                    continue
                s = str(val)
                print(f"{tagname}: len={len(s)}  head:\n{s[:max_chars]}")
                # quick “do we have the big SI header?”
                hits = re.findall(r"(?m)^(SI\.[^\s=]+)\s*=", s)
                if hits:
                    print(f"Found {len(hits)} SI.* keys, e.g.: {hits[:15]}")
                else:
                    print("Found 0 SI.* keys in this tag.")

        show(p0, "PAGE 0")
        show(plast, "LAST PAGE")

# dump_scanimage_tags("/Users/jcoleman/Documents/--LARGE DATA--/#Khodayari/260113_NK/process1/fileCELL_3x_750nm_00004.tif")
dump_scanimage_tags(tif_file)


In [8]:
import tifffile as tiff
import re

def find_si_anywhere(tif_path, pages_to_check=200):
    with tiff.TiffFile(tif_path) as tf:
        n = len(tf.pages)
        pages = list(range(min(pages_to_check, n))) + ([n-1] if n-1 >= pages_to_check else [])
        pages = sorted(set(pages))

        for i in pages:
            page = tf.pages[i]
            for tagname in ["Artist", "Software", "ImageDescription"]:
                tag = page.tags.get(tagname)
                if not tag:
                    continue
                val = tag.value
                # decode bytes safely
                if isinstance(val, (bytes, bytearray)):
                    s = val.decode("utf-8", errors="ignore")
                else:
                    s = str(val)

                if "SI." in s:
                    hits = re.findall(r"(?m)^(SI\.[^\s=]+)\s*=", s)
                    print(f"FOUND on page {i} tag {tagname}: SI. present, {len(hits)} keys (sample: {hits[:10]})")
                    return i, tagname, s[:4000]

        print("No 'SI.' found in Artist/Software/ImageDescription for checked pages.")
        return None, None, None

find_si_anywhere(tif_file)


FOUND on page 0 tag Software: SI. present, 441 keys (sample: ['SI.LINE_FORMAT_VERSION', 'SI.PREMIUM', 'SI.TIFF_FORMAT_VERSION', 'SI.VERSION_COMMIT', 'SI.VERSION_MAJOR', 'SI.VERSION_MINOR', 'SI.VERSION_UPDATE', 'SI.acqState', 'SI.acqsPerLoop', 'SI.errorMsg'])


(0,
 'Software',
 "SI.LINE_FORMAT_VERSION = 1\nSI.PREMIUM = true\nSI.TIFF_FORMAT_VERSION = 4\nSI.VERSION_COMMIT = '6fa08573b737b7ca5870c37cdb5781839365bdf6'\nSI.VERSION_MAJOR = 2023\nSI.VERSION_MINOR = 1\nSI.VERSION_UPDATE = 1\nSI.acqState = 'grab'\nSI.acqsPerLoop = 1\nSI.errorMsg = ''\nSI.extTrigEnable = false\nSI.fieldCurvatureRxs = []\nSI.fieldCurvatureRys = []\nSI.fieldCurvatureTilt = 0\nSI.fieldCurvatureTip = 0\nSI.fieldCurvatureZs = []\nSI.hBeams.enablePowerBox = false\nSI.hBeams.errorMsg = ''\nSI.hBeams.flybackBlanking = true\nSI.hBeams.interlaceDecimation = [1 1]\nSI.hBeams.interlaceOffset = [0 0]\nSI.hBeams.lengthConstants = [Inf Inf]\nSI.hBeams.name = 'SI Beams'\nSI.hBeams.powerBoxEndFrame = Inf\nSI.hBeams.powerBoxStartFrame = 1\nSI.hBeams.powerFractionLimits = [1 1]\nSI.hBeams.powerFractions = [0.51 0]\nSI.hBeams.powers = [51 0]\nSI.hBeams.pzAdjust = [scanimage.types.BeamAdjustTypes.Exponential scanimage.types.BeamAdjustTypes.Exponential]\nSI.hBeams.pzFunction = {'@scanimage

In [2]:
# Output image metadata to csv- and/or txt-friendly format

from pathlib import Path
import json
import socket
import platform
import re
import pandas as pd
import tifffile as tiff
from datetime import datetime

def _parse_epoch_line(desc_text: str):
    """
    Parse ScanImage epoch line like:
      epoch = [2026  1 13 16 24 29.809]
    Returns datetime or None.
    """
    if not desc_text:
        return None, None

    m = re.search(r"(?m)^epoch\s*=\s*\[([^\]]+)\]\s*$", desc_text)
    if not m:
        return None, None

    raw = m.group(1).strip()  # "2026  1 13 16 24 29.809"
    parts = [p for p in re.split(r"\s+", raw) if p]
    if len(parts) < 6:
        return None, raw

    y = int(float(parts[0]))
    mo = int(float(parts[1]))
    d = int(float(parts[2]))
    h = int(float(parts[3]))
    mi = int(float(parts[4]))
    sec = float(parts[5])

    sec_int = int(sec)
    usec = int(round((sec - sec_int) * 1e6))
    dt = datetime(y, mo, d, h, mi, sec_int, usec)
    return dt, raw

def _get_si_value(software_text: str, key: str):
    """
    Extract a ScanImage 'SI.key = value' line from the Software tag text.
    Returns the value string (stripped) or None.
    """
    if not software_text:
        return None
    m = re.search(rf"(?m)^{re.escape(key)}\s*=\s*(.*)$", software_text)
    return m.group(1).strip() if m else None

def _parse_si_vector(txt):
    if not txt:
        return None
    s = txt.strip().strip("[]")
    parts = [p for p in s.replace(",", " ").split() if p]
    try:
        return [float(p) for p in parts]
    except ValueError:
        return None

def fov_um_from_tiff_resolution(page):
    tags = page.tags

    # XResolution/YResolution are (num, denom) tuples
    res_x_num, res_x_den = tags["XResolution"].value
    res_y_num, res_y_den = tags["YResolution"].value
    res_x = res_x_num / res_x_den  # pixels per unit
    res_y = res_y_num / res_y_den

    unit = tags["ResolutionUnit"].value  # 2=inches, 3=cm

    height, width = page.shape  # rows, cols

    if unit == 3:  # cm
        width_um  = (width  / res_x) * 1e4
        height_um = (height / res_y) * 1e4
    elif unit == 2:  # inch
        width_um  = (width  / res_x) * 25400
        height_um = (height / res_y) * 25400
    else:
        return None, None, None, None  # unsupported / missing

    um_per_px_x = width_um / width
    um_per_px_y = height_um / height
    
    return width_um, height_um, um_per_px_x, um_per_px_y


def export_scanimage_metadata(tif_path, out_csv=None):
    tif_path = Path(tif_path)

    with tiff.TiffFile(tif_path) as tf:
        p0 = tf.pages[0]
        
        # FOV fallback from TIFF resolution tags (matches what Fiji reports)
        fov_x_um_tiff, fov_y_um_tiff, um_per_pixel_x_tiff, um_per_pixel_y_tiff = fov_um_from_tiff_resolution(p0)


        software = str(p0.tags.get("Software").value) if p0.tags.get("Software") else ""
        artist = str(p0.tags.get("Artist").value) if p0.tags.get("Artist") else ""
        desc = str(p0.tags.get("ImageDescription").value) if p0.tags.get("ImageDescription") else ""

        imaging_fov_um = _get_si_value(software, "SI.hRoiManager.imagingFovUm")

        # ---------- File info ----------
        stat = tif_path.stat()
        file_created = datetime.fromtimestamp(stat.st_ctime)
        file_modified = datetime.fromtimestamp(stat.st_mtime)

        # ---------- Acquisition epoch -> datetime ----------
        acq_dt, acq_epoch_raw = _parse_epoch_line(desc)

        # # ---------- ROI geometry (Artist tag JSON) ----------
        # fov_x_um = fov_y_um = px_x = px_y = None
        # scan_zoom_factor = None
        # try:
        #     roi_root = json.loads(artist)
        #     roi = roi_root["RoiGroups"]["imagingRoiGroup"]["rois"]
        #     scanfield = roi["scanfields"]
        #     # fov_x_um, fov_y_um = scanfield["sizeXY"]
        #     fov_xy = _parse_si_vector(imaging_fov_um)

        #     if fov_xy and len(fov_xy) == 2:
        #         fov_x_um, fov_y_um = fov_xy
        #         um_per_pixel_x = fov_x_um / px_x
        #         um_per_pixel_y = fov_y_um / px_y
        #     else:
        #         fov_x_um = fov_y_um = um_per_pixel_x = um_per_pixel_y = None

        #     px_x, px_y = scanfield["pixelResolutionXY"]
        #     scan_zoom_factor = roi.get("UserData", {}).get("scanZoomFactor", None)

        # except Exception:
        #     # If Artist isn't valid JSON, just leave these as None
        #     pass
        # ---------- ROI geometry (Artist tag JSON) ----------
        fov_x_um = fov_y_um = px_x = px_y = None
        um_per_pixel_x = um_per_pixel_y = None
        scan_zoom_factor = None
        
        try:
            roi_root = json.loads(artist)
            roi = roi_root["RoiGroups"]["imagingRoiGroup"]["rois"]
            scanfield = roi["scanfields"]
        
            # pixel dims (needed for um/px)
            px_x, px_y = scanfield["pixelResolutionXY"]
        
            # calibrated FOV from Software tag (preferred)
            fov_xy = _parse_si_vector(imaging_fov_um)
            if fov_xy and len(fov_xy) == 2 and px_x and px_y:
                fov_x_um, fov_y_um = fov_xy
                um_per_pixel_x = fov_x_um / px_x
                um_per_pixel_y = fov_y_um / px_y
        
            scan_zoom_factor = roi.get("UserData", {}).get("scanZoomFactor", None)
        
        except Exception:
            pass
            
        # Fallback if ScanImage FOV is missing
        if fov_x_um is None and fov_x_um_tiff is not None:
            fov_x_um = fov_x_um_tiff
            fov_y_um = fov_y_um_tiff
            um_per_pixel_x = um_per_pixel_x_tiff
            um_per_pixel_y = um_per_pixel_y_tiff
            fov_source = "tiff_resolution_tags"
        else:
            fov_source = "scanimage_imagingFovUm" if fov_x_um is not None else "missing"

        # ---------- Basic image geometry from TIFF page ----------
        # tifffile pages have shape, but tags are reliable:
        image_width_px = getattr(p0, "imagewidth", None)
        image_height_px = getattr(p0, "imagelength", None)

        # imaging_fov_um = _get_si_value(software, "SI.hRoiManager.imagingFovUm")

        # ---------- ScanImage version / files ----------
        ver_major = _get_si_value(software, "SI.VERSION_MAJOR")
        ver_minor = _get_si_value(software, "SI.VERSION_MINOR")
        ver_update = _get_si_value(software, "SI.VERSION_UPDATE")

        scanimage_version = None
        if ver_major and ver_minor and ver_update:
            scanimage_version = f"{ver_major}.{ver_minor}.{ver_update}"

        # ---------- Channels / optics ----------
        channels_saved = _get_si_value(software, "SI.hChannels.channelSave")
        channel_names = _get_si_value(software, "SI.hChannels.channelName")
        frame_rate_hz = _get_si_value(software, "SI.hRoiManager.scanFrameRate")

        # NOTE: In your screenshot + file dump, laser wavelength is from the laser UI,
        # but ScanImage often stores it as part of beam "powers" or separate fields depending on integration.
        # We'll still report beam powers & power fractions when present.
        beam_power_mw = _get_si_value(software, "SI.hBeams.powers")
        beam_power_fraction = _get_si_value(software, "SI.hBeams.powerFractions")

        # PMT gains are controller-specific; may be missing in some configs
        pmt_gains = _get_si_value(software, "SI.hPmts.gain")
        pmt_offsets = _get_si_value(software, "SI.hChannels.channelOffset")

        # ---------- Scanner type ---------- #begin_new1
        scanner_name = (_get_si_value(software, "SI.hScan2D.scannerType")
                        or _get_si_value(software, "SI.hScan2D.name"))
        scanner_resonant = _get_si_value(software, "SI.hScan2D.resonantScannerEnable")
        # scanner_bidirectional = _get_si_value(software, "SI.hScan2D.bidirectionalScan")
        line_rate_hz = _get_si_value(software, "SI.hScan2D.scanLineRate")

        # Scanner classification
        scanner_resonant_enabled = (scanner_name == "'RGG'")
        
        # Resonant scanners are inherently bidirectional
        scanner_bidirectional = True if scanner_resonant_enabled else _get_si_value(
            software, "SI.hScan2D.bidirectionalScan"
        )
        
        # Line rate is not logged for resonant scanners
        scan_line_rate_hz = None
        scan_line_rate_note = (
            "Resonant scanner line rate is hardware-defined and not written to TIFF metadata"
            if scanner_resonant_enabled else "Derived from ScanImage metadata"
        )
        # Can manually derive from fps value and pixels
        512/30
        Out[2]: 17.066666666666666
        
        17.067*30
        Out[5]: 512.01
        
        17.0667*512
        Out[6]: 8738.1504
        
        # ---------- Pockels / power ----------
        pockels_power_fractions = _get_si_value(software, "SI.hBeams.powerFractions")
        pockels_power_limits = _get_si_value(software, "SI.hBeams.powerFractionLimits")
        pockels_lut_source = _get_si_value(software, "SI.hBeams.pzLUTSource")
        pockels_pz_adjust = _get_si_value(software, "SI.hBeams.pzAdjust")
        pockels_flyback_blanking = _get_si_value(software, "SI.hBeams.flybackBlanking")
        pockels_turnaround_blanking = _get_si_value(software, "SI.hBeams.turnAroundBlanking")
        
        # If your config logs absolute mW in SI.hBeams.powers, keep that too:
        beam_powers_mw = _get_si_value(software, "SI.hBeams.powers")
        
        # ---------- PMT values ----------
        # Gains are sometimes here; offsets often under hChannels
        pmt_gains = (_get_si_value(software, "SI.hPmts.gain")
                     or _get_si_value(software, "SI.hPmts.gains"))
        pmt_offsets = (_get_si_value(software, "SI.hChannels.channelOffset")
                       or _get_si_value(software, "SI.hPmts.offset"))
        
        # ---------- Frames averaged / averaging ----------
        # Display rolling average is NOT acquisition averaging; you want these when present:
        log_avg_enable = _get_si_value(software, "SI.hScan2D.logAverageEnable")
        log_avg_factor = _get_si_value(software, "SI.hScan2D.logAverageFactor")
        
        # Also useful: scan/recording averaging modes can vary across versions:
        # (These might not exist in every config; safe to keep.)
        scan_avg_enable = _get_si_value(software, "SI.hScan2D.scanAverageEnable")
        scan_avg_factor = _get_si_value(software, "SI.hScan2D.scanAverageFactor") #end new1

        # Objective name exists only if the objective manager is configured
        objective_name = _get_si_value(software, "SI.hObjectiveManager.objectiveName")

        # ---------- TIFF structure ----------
        num_frames = len(tf.pages)
        timestamps_present = "frameTimestamps_sec" in desc

        # ---------- Derived um/px ----------
        um_per_pixel_x = (float(fov_x_um) / float(px_x)) if (fov_x_um and px_x) else None
        um_per_pixel_y = (float(fov_y_um) / float(px_y)) if (fov_y_um and px_y) else None

        row = {
            # File / identity
            "filename": tif_path.name,
            "full_path": str(tif_path),
            "file_size_mb": round(stat.st_size / 1e6, 2),
            "file_created_time": file_created.isoformat(),
            "file_modified_time": file_modified.isoformat(),
            "acquisition_datetime": acq_dt.isoformat() if acq_dt else None,
            "acquisition_epoch_raw": acq_epoch_raw,
            "scanimage_version": scanimage_version,
            "scanimage_commit": _get_si_value(software, "SI.VERSION_COMMIT"),
            "scanimage_premium": _get_si_value(software, "SI.PREMIUM"),
            "config_file": _get_si_value(software, "SI.hConfigurationSaver.cfgFilename"),
            "user_file": _get_si_value(software, "SI.hConfigurationSaver.usrFilename"),
            "host_os": platform.platform(),
            "host_machine": socket.gethostname(),
            "fov_source": fov_source,

            # Image geometry
            "pixels_per_line": px_x,
            "lines_per_frame": px_y,
            "image_width_px": image_width_px,
            "image_height_px": image_height_px,
            "fov_x_um": fov_x_um,
            "fov_y_um": fov_y_um,
            "um_per_pixel_x": um_per_pixel_x,
            "um_per_pixel_y": um_per_pixel_y,

            # Acquisition structure
            "num_frames": num_frames,
            "frame_rate_hz": frame_rate_hz,
            "timestamps_present": timestamps_present,
            "channels_saved": channels_saved,
            "channel_names": channel_names,
            "bit_depth": p0.bitspersample if hasattr(p0, "bitspersample") else None,
            "samples_per_pixel": getattr(p0, "samplesperpixel", None),
            "planar_configuration": getattr(p0, "planarconfiguration", None),

            # Scanner
            # "scanner_name": scanner_name,
            # "scanner_resonant_enabled": scanner_resonant,
            # "scanner_bidirectional": scanner_bidirectional,
            # "scan_line_rate_hz": line_rate_hz,
            "scanner_name": scanner_name,
            "scanner_resonant_enabled": scanner_resonant_enabled,
            "scanner_bidirectional": scanner_bidirectional,
            "scan_line_rate_hz": scan_line_rate_hz,
            "scan_line_rate_note": scan_line_rate_note,
            
            # Pockels / power
            "pockels_power_fractions": pockels_power_fractions,
            "pockels_power_limits": pockels_power_limits,
            "pockels_lut_source": pockels_lut_source,
            "pockels_pz_adjust": pockels_pz_adjust,
            "pockels_flyback_blanking": pockels_flyback_blanking,
            "pockels_turnaround_blanking": pockels_turnaround_blanking,
            "beam_powers_mw": beam_powers_mw,
            
            # PMTs
            "pmt_gains": pmt_gains,
            "pmt_offsets": pmt_offsets,
            
            # Averaging (acquisition/log averaging when present)
            "log_avg_enable": log_avg_enable,
            "log_avg_factor": log_avg_factor,
            "scan_avg_enable": scan_avg_enable,
            "scan_avg_factor": scan_avg_factor,

            # Optics / laser
            "objective_name": objective_name,
            "scan_zoom_factor": scan_zoom_factor,
            "beam_power_mw": beam_power_mw,
            "beam_power_fraction": beam_power_fraction,
            "pmt_offsets": pmt_offsets,
            "pmt_gains": pmt_gains,

            # Z semantics (explicit & honest)
            "acquisition_type": "single_plane",
            "z_logged": "no",
            "z_reference": "roi_relative_zero",
            "z_note": "Single-plane T-series; no StackManager/FastZ Z positions logged into TIFF",
        }

    print("TIFF resolution tags FOV:", fov_x_um_tiff, "*", fov_y_um_tiff, "um;", "um/px:", "x=", um_per_pixel_x_tiff, "y=", um_per_pixel_y_tiff)

    df = pd.DataFrame([row])

    df_kv = df.T.reset_index()
    df_kv.columns = ["field", "value"]


    if out_csv is None:
        out_csv = tif_path.with_suffix("").as_posix() + "_metadata.csv"
    df_kv.to_csv(out_csv, index=False)
    return df_kv


In [5]:
si_meta = export_scanimage_metadata(
    tif_file
)

TIFF resolution tags FOV: 162.99724578857422 * 162.99724578857422 um; um/px: x= 0.318353995680809 y= 0.318353995680809


In [6]:
needles = ["hbeams", "pockels", "power", "beam", "lut", "fastz", "zstep", "zs", "stack", "motor"]
for n in needles:
    hits = [k for k in si_meta.keys() if n in k.lower()]
    print(n, len(hits))
    for k in hits[:30]:
        print("  ", k)

hbeams 0
pockels 0
power 0
beam 0
lut 0
fastz 0
zstep 0
zs 0
stack 0
motor 0


In [ ]:
# Annotate stack images with Z position (eg top right corner in microns (actual to compare across